# 13.a — Ad hoc test: suitable land and the effect of the AMOC weakening on the land available for cropping

**Ad hoc test, outside the suitability chain**: it reads the final suitability of 6.b (`climate_soil_scenarios_<crop>.nc`,
ERA5 and the four runs), the ESA CCI Land Cover map and the World Database on Protected Areas, and writes only in
`<out_root>/test_land_availability/`. The model and its maps are unchanged: land use never enters the model, the
availability is a reporting layer applied at the end.

**Question.** 6.b reports the suitable land on all land (climate and soil only). Part of the land that becomes suitable
or unsuitable is protected, built up or forest, where cropping is not possible or not wanted anyway. How do the suitable
areas, the land gained and lost, and the effect of the AMOC weakening (hosing − no hosing) change on the land available
for cropping?

**Method.**
- **Exclusion layers at 300 m**, counted per EC-Earth3 cell as fractions of the cell's land pixels (as 0.1.b; never a
  binary flag per cell):
  - *protected*: WDPA polygons with status designated, inscribed or established, without the UNESCO-MAB biosphere
    reserves and the marine-only areas (the selection of the Protected Planet coverage statistics); the sites known only
    as a point (no boundary) are drawn, as by Protected Planet, as a circle of their reported land area around the point
    (`WDPA_POINTS`); *strict* = IUCN categories Ia, Ib and II only (categories V–VI include farmed landscapes);
  - *built-up*: ESA CCI class 190;
  - *tree cover*: ESA CCI classes 50–90 (tree cover, closed to open, > 15 %) and 160–170 (flooded tree cover); the
    mosaics (100) are not counted.
- **Availability variants** (fraction of the cell's land that can be counted):
  1. *all land* = 6.b (check: reproduces its tables);
  2. *without protected and built-up* — the main variant: legal and physical exclusions;
  3. *without strict protected (IUCN Ia–II) and built-up* — lighter;
  4. *without protected, built-up and tree cover* — "no deforestation", a societal choice, a sensitivity only.
- **Areas**: suitable land = Σ land area × available fraction over the suitable cells; the same metrics as 6.b: change
  of the suitable land with the hosing on all land and on the cells assessable in both runs (%), land gained and lost
  (Mha), change per region; also each run against ERA5 (the warming).
- **Where the land gained and lost lies**: split into available, protected, built-up and tree cover (not protected).
- *Today's crop area* (share of the MIRCA2000 area of the crop in suitable cells) does not change: cultivated land is
  available by definition.

**Reading.** If the effect of the AMOC weakening in % changes little between the variants, it is not driven by land
that could not be cropped anyway: the potential (all land) stays the main result and the available land is given as
context. If the gains or losses lie mostly in protected land or forest, the gross areas of 6.b overstate the land that
could actually change use, and the available values should be quoted next to them.

**Data.**
- ESA CCI Land Cover v2.0.7 (BDAP `base/Landcover/GLOBAL/ESA-CCI-LC`), year 2015 by default: dataset
  `esa_cci_lc_2015` of 00 (check the year files present with `ls` and change the pattern in 00 if needed).
- WDPA: BDAP `base/ProtectedSites/GLOBAL/WDPA/LATEST/Data/FileGDB` (version of 2019): dataset `wdpa` of 00. For a newer version, download
  the file geodatabase of the month from https://www.protectedplanet.net/en/thematic-areas/wdpa (Download → File
  Geodatabase; terms of use: non-commercial, cite UNEP-WCMC and IUCN) and unzip it into
  `/eos/jeodpp/data/projects/DROUGHTOBS/elisa/Crop_suitability/WDPA/`: dataset `wdpa` of 00 (`WDPA_*_Public.gdb`).
- Run 00 again after adding the data, then this notebook (the 300 m counting takes a while; it is cached in
  `land_availability_ecearth3.nc` and read back while the inputs and the classes are unchanged).

**Limits.** The layers are static (protected areas of today, land cover of 2015) and applied unchanged to 2025 and
2050; the circles of the WDPA points are approximate locations; pixels are assigned by their centre.

Outputs in `<out_root>/test_land_availability/`: `land_availability_ecearth3.nc` (fractions per cell), and in
`<route>/<set>/`: `areas_availability.csv`, `amoc_effect_availability.csv`, `amoc_effect_regions_availability.csv`,
`gained_lost_by_land_category.csv`, the figures.

In [ ]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import time
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu
CFG = cu.load_config()

# ---- parameters: the final set of 6.b
CHOSEN_VARIANT = "A_60d"
CHOSEN_RAD = os.environ.get("CHOSEN_RAD", "no_rad")   # as 6.b
CELL_AREA_THR = 0.0005
CROPS = ["Maize_rf", "Potato_rf", "Soybean_rf", "Rice_rf", "Wheat_rf", "Wheat-winter_rf", "Wheat-spring_rf"]
RUNS = ["gcm_2025_nohosing", "gcm_2025_hosing", "gcm_2050_nohosing", "gcm_2050_hosing"]
PAIRS = [("gcm_2025_hosing", "gcm_2025_nohosing"), ("gcm_2050_hosing", "gcm_2050_nohosing")]   # hosing - original
LEVELS = ["low", "high"]

# ---- exclusion layers
BUILT_CLASSES = [190]                                                    # ESA CCI urban areas
TREE_CLASSES = [50, 60, 61, 62, 70, 71, 72, 80, 81, 82, 90, 160, 170]   # tree cover > 15 % (closed to open), flooded tree cover
NOT_LAND = [0, 210]                                                      # no data, water (as 0.1.b)
WDPA_STATUS = ["designated", "inscribed", "established"]                # Protected Planet coverage statistics
STRICT_IUCN = ["Ia", "Ib", "II"]
WDPA_POINTS = True            # sites without boundary drawn as circles of their reported land area (Protected Planet)
SIMPLIFY_DEG = 0.001          # simplification of the WDPA polygons (~100 m, below the 300 m pixel); 0 = none
BLOCK = 900                   # rows of the 300 m map processed at once (900 x 129600 pixels)
FORCE = os.environ.get("LANDAV_FORCE", "0") == "1"   # recount the 300 m layers even if the cache matches

SET = f"{CHOSEN_VARIANT}_{CHOSEN_RAD}"
ROUTE = cu.route(CELL_AREA_THR)
B6 = Path(os.environ.get("LANDAV_6B") or cu.out_dir(CFG, "6_climate_soil_amoc", ROUTE, SET))   # read only
LC_FILE = Path(os.environ.get("LANDAV_LC") or cu.data_file(CFG, "esa_cci_lc_2015", "lccs_class"))
WDPA_FILE = Path(os.environ.get("LANDAV_WDPA") or cu.data_file(CFG, "wdpa"))
OUT0 = Path(os.environ.get("LANDAV_OUT") or cu.out_dir(CFG, "test_land_availability"))   # the only folder written
OUT = OUT0 / ROUTE / SET
OUT.mkdir(parents=True, exist_ok=True)
SCEN = ["ERA5", *RUNS]

GRID = cu.svm_grid(CFG)
NLAT, NLON = GRID["lat"].size, GRID["lon"].size
LAND = cu.land_mask(CFG).values
LAND_MHA = np.nan_to_num(cu.land_area_ha(CFG)) * LAND / 1e6
LAT2, LON2 = np.meshgrid(GRID["lat"], GRID["lon"], indexing="ij")
LON180 = np.mod(LON2 + 180, 360) - 180
REGIONS = {"global": LAND,                                                   # as 6.b
           "Europe": LAND & (LAT2 >= 35) & (LAT2 <= 72) & (LON180 >= -25) & (LON180 <= 45),
           "north of 50 N": LAND & (LAT2 > 50),
           "23.5 - 50 N": LAND & (LAT2 > 23.5) & (LAT2 <= 50),
           "tropics": LAND & (np.abs(LAT2) <= 23.5),
           "south of 23.5 S": LAND & (LAT2 < -23.5)}
mha = lambda w: float(np.nansum(LAND_MHA * w))
for f in (LC_FILE, WDPA_FILE, B6):
    assert f.exists(), f"not found: {f} (see Data above, then run 00 again)"
print(f"set {SET} | 6.b {B6}\nland cover {LC_FILE.name} | WDPA {WDPA_FILE.name}\nwritten only in {OUT0}")

## 1. Exclusion layers: protected, built-up and tree cover per EC-Earth3 cell

The WDPA polygons are burnt on the 300 m grid of the land cover, block by block (strict categories last, so that a pixel
covered by a strict and a non-strict area counts as strict); each pixel is then counted in its EC-Earth3 cell. The
categories are counted without overlap: protected (any), built-up not protected, tree cover neither protected nor
built-up.

In [ ]:
def wdpa_read(path, key, extra=()):
    # one WDPA layer ("poly" or "point") with the filters of the coverage: status, no UNESCO-MAB, no marine-only
    import geopandas as gpd
    import pyogrio
    layers = [str(l[0]) for l in pyogrio.list_layers(path)]
    layer = next((l for l in layers if key in l.lower()), None)
    if layer is None:
        print(f"WDPA: no {key} layer in {layers}")
        return None
    fields = list(pyogrio.read_info(path, layer=layer)["fields"])
    col = {c.upper(): c for c in fields}
    need = ["STATUS", "DESIG_ENG", "MARINE", "IUCN_CAT", *extra]
    assert all(c in col for c in need), f"WDPA layer {layer}: columns {need} expected, found {fields}"
    g = gpd.read_file(path, layer=layer, columns=[col[c] for c in need], force_2d=True)
    n0 = len(g)
    g = g.rename(columns={col[c]: c for c in need})
    keep = g["STATUS"].astype(str).str.strip().str.lower().isin(WDPA_STATUS)
    keep &= ~g["DESIG_ENG"].astype(str).str.contains("UNESCO-MAB", case=False, na=False)
    keep &= g["MARINE"].astype(str).str.strip() != "2"
    g = g[keep & g.geometry.notna() & ~g.geometry.is_empty]
    g = g.to_crs(4326) if g.crs is not None and g.crs.to_epsg() != 4326 else g
    print(f"WDPA layer {layer}: {n0} features, {len(g)} kept", flush=True)
    return g


def wdpa_polygons(path):
    # polygons, and the points as circles of their reported land area (REP_AREA - REP_M_AREA, km2; a multi-point site
    # shares it between its points); value 2 = strict IUCN, 1 = other
    import geopandas as gpd
    import shapely
    from shapely import affinity
    t0 = time.time()
    g = wdpa_read(path, "poly")[["IUCN_CAT", "geometry"]]
    if SIMPLIFY_DEG > 0:
        g["geometry"] = g.geometry.simplify(SIMPLIFY_DEG, preserve_topology=True)
    parts = [g]
    if WDPA_POINTS:
        pt = wdpa_read(path, "point", ("REP_AREA", "REP_M_AREA"))
        if pt is not None and len(pt):
            km2 = (pd.to_numeric(pt["REP_AREA"], errors="coerce").fillna(0)
                   - pd.to_numeric(pt["REP_M_AREA"], errors="coerce").fillna(0)).clip(lower=0)
            pt = pt.assign(km2=km2.values / shapely.get_num_geometries(pt.geometry.values))
            pt = pt[pt.km2 > 0].explode(index_parts=False)
            r = np.sqrt(pt.km2.values / np.pi) / 111.32                       # radius, degrees of latitude
            circ = [affinity.scale(q.buffer(1.0, 16), xfact=ri / max(np.cos(np.radians(q.y)), 0.05), yfact=ri, origin=q)
                    for q, ri in zip(pt.geometry.values, r)]
            parts.append(gpd.GeoDataFrame({"IUCN_CAT": pt["IUCN_CAT"].values}, geometry=circ, crs=4326))
            print(f"WDPA points: {len(pt)} circles, reported land area {pt.km2.sum() / 1e6:.2f} million km2", flush=True)
    g = pd.concat(parts, ignore_index=True)
    strict = g["IUCN_CAT"].astype(str).str.strip().isin(STRICT_IUCN)
    g = g.assign(val=np.where(strict, 2, 1).astype("uint8")).sort_values("val", kind="stable").reset_index(drop=True)
    print(f"WDPA: {len(g)} shapes ({int((g.val == 2).sum())} strict), {(time.time() - t0) / 60:.1f} min", flush=True)
    return g[["val", "geometry"]]


SPEC = {"lc_file": LC_FILE.name, "wdpa_file": WDPA_FILE.name, "built": str(BUILT_CLASSES), "tree": str(TREE_CLASSES),
        "not_land": str(NOT_LAND), "wdpa_status": str(WDPA_STATUS), "strict_iucn": str(STRICT_IUCN),
        "simplify_deg": SIMPLIFY_DEG, "wdpa_points": str(WDPA_POINTS)}
CACHE = OUT0 / "land_availability_ecearth3.nc"
FR = None
if CACHE.exists() and not FORCE:
    with xr.open_dataset(CACHE) as ds:
        if all(str(ds.attrs.get(k)) == str(v) for k, v in SPEC.items()):
            FR = {v: ds[v].values for v in ds.data_vars}
            print("read back:", CACHE)
if FR is None:
    from rasterio import features
    from rasterio.transform import from_origin
    from shapely.geometry import box
    G = wdpa_polygons(WDPA_FILE)
    SIDX = G.sindex
    t0 = time.time()
    ds = xr.open_dataset(LC_FILE)
    lc = ds["lccs_class"]
    lc = lc.isel(time=0) if "time" in lc.dims else lc
    lat, lon = ds["lat"].values, ds["lon"].values
    assert np.all(np.diff(lon) > 0), "longitudes of the land cover must be ascending"
    dx, dy = abs(lon[1] - lon[0]), abs(lat[1] - lat[0])
    desc = lat[0] > lat[-1]
    I_row, _ = cu.cell_of(lat, np.zeros_like(lat), GRID)          # EC-Earth3 row of each 300 m row
    _, J_col = cu.cell_of(np.zeros_like(lon), lon, GRID)          # EC-Earth3 column of each 300 m column
    NAMES = ["land", "protected", "protected_strict", "built_np", "built_nps", "tree_np"]
    cnt = {k: np.zeros((NLAT, NLON)) for k in NAMES}
    for r0 in range(0, lat.size, BLOCK):
        x = np.asarray(lc.isel(lat=slice(r0, r0 + BLOCK)).values)
        la = lat[r0:r0 + x.shape[0]]
        top, bottom = la.max() + dy / 2, la.min() - dy / 2
        idx = np.sort(SIDX.query(box(lon[0] - dx / 2, bottom, lon[-1] + dx / 2, top)))   # sorted: strict drawn last
        prot = np.zeros(x.shape, np.uint8)
        if idx.size:
            prot = features.rasterize(zip(G.geometry.values[idx], G["val"].values[idx]), out_shape=x.shape,
                                      transform=from_origin(lon[0] - dx / 2, top, dx, dy), fill=0, dtype="uint8")
            prot = prot if desc else prot[::-1]                   # rasterize writes north up
        land = ~np.isin(x, NOT_LAND)
        p_any, p_str = (prot > 0) & land, (prot == 2) & land
        built = np.isin(x, BUILT_CLASSES) & land
        m = {"land": land, "protected": p_any, "protected_strict": p_str, "built_np": built & ~p_any,
             "built_nps": built & ~p_str, "tree_np": np.isin(x, TREE_CLASSES) & land & ~p_any & ~built}
        rows_i = I_row[r0:r0 + x.shape[0]]
        for i in np.unique(rows_i):
            sel = rows_i == i
            for k in NAMES:
                cnt[k][i] += np.bincount(J_col, weights=m[k][sel].sum(axis=0), minlength=NLON)
        if r0 // BLOCK % 8 == 0:
            print(f"  rows {r0 + x.shape[0]} / {lat.size} ({(time.time() - t0) / 60:.1f} min)", flush=True)
    ds.close()
    n = np.maximum(cnt["land"], 1)
    FR = {"esa_land_pixels": cnt["land"], **{f"frac_{k}": cnt[k] / n for k in NAMES[1:]}}
    long = {"esa_land_pixels": "number of land pixels of the land cover (300 m) in the cell",
            "frac_protected": "share of the land pixels in a protected area (WDPA, kept for the coverage)",
            "frac_protected_strict": "same, IUCN categories Ia, Ib and II only",
            "frac_built_np": "built-up (ESA CCI 190), not protected",
            "frac_built_nps": "built-up, not strictly protected",
            "frac_tree_np": "tree cover (ESA CCI 50-90, 160-170), neither protected nor built-up"}
    xr.Dataset({k: (("lat", "lon"), v.astype("float32"), {"long_name": long[k]}) for k, v in FR.items()},
               coords={"lat": GRID["lat"], "lon": GRID["lon"]}, attrs=SPEC).to_netcdf(CACHE)
    print(f"done in {(time.time() - t0) / 60:.1f} min, written {CACHE}")

FR = {k: np.where(LAND, v, 0.0) for k, v in FR.items()}
NO_ESA = LAND & (FR["esa_land_pixels"] == 0)      # land of the SVM grid without land pixels in the land cover: all available
AV = {"all land": np.where(LAND, 1.0, 0.0),
      "without protected and built-up": np.clip(1 - FR["frac_protected"] - FR["frac_built_np"], 0, 1) * LAND,
      "without strict protected and built-up": np.clip(1 - FR["frac_protected_strict"] - FR["frac_built_nps"], 0, 1) * LAND,
      "without protected, built-up and tree cover": np.clip(1 - FR["frac_protected"] - FR["frac_built_np"]
                                                            - FR["frac_tree_np"], 0, 1) * LAND}
CATS = {"available": AV["without protected, built-up and tree cover"], "protected": FR["frac_protected"],
        "built-up": FR["frac_built_np"], "tree cover (not protected)": FR["frac_tree_np"]}   # sum to 1 on land
print(f"land cells without land-cover land pixels: {int(NO_ESA.sum())} ({mha(NO_ESA):.1f} Mha, counted available)")

## 2. The layers: shares of the land per region, and maps

Sanity check: WDPA covers about 17–18 % of the land of the world in the Protected Planet statistics (polygons and
points, all land, in 2024); the version of 2019 used here should give somewhat less.

In [ ]:
rows = []
for reg, rm in REGIONS.items():
    land = mha(rm)
    rows.append({"region": reg, "land (Mha)": land,
                 "protected (%)": 100 * mha(rm * FR["frac_protected"]) / land,
                 "strict protected (%)": 100 * mha(rm * FR["frac_protected_strict"]) / land,
                 "built-up, not protected (%)": 100 * mha(rm * FR["frac_built_np"]) / land,
                 "tree cover, not protected (%)": 100 * mha(rm * FR["frac_tree_np"]) / land,
                 **{f"available: {a} (%)": 100 * mha(rm * w) / land for a, w in AV.items() if a != "all land"}})
LAYERS = pd.DataFrame(rows).set_index("region")
LAYERS.to_csv(OUT0 / "land_availability_regions.csv")
display(LAYERS.round(1))

coords = {"lat": GRID["lat"], "lon": GRID["lon"]}
fig, axes = plt.subplots(2, 1, figsize=(15, 13))
for ax, a in zip(axes, ["without protected and built-up", "without protected, built-up and tree cover"]):
    xr.DataArray(np.where(LAND, AV[a], np.nan), coords=coords, dims=("lat", "lon")).plot(
        ax=ax, cmap="YlGn", vmin=0, vmax=1, cbar_kwargs={"label": "available share of the land of the cell"})
    ax.set_title(f"Land available for cropping: {a} (WDPA, ESA CCI {LC_FILE.name.split('P1Y-')[-1][:4]})")
plt.tight_layout(); plt.savefig(OUT0 / "land_availability_maps.png", dpi=90); plt.show()

## 3. Suitable land and the effect of the AMOC weakening on the available land

The tables of 6.b recomputed with each availability variant: suitable land per scenario, and for each pair (hosing −
no hosing, and each run − ERA5) the change of the suitable land on all land and on the cells assessable in both
(novel in neither), the land gained and lost, and the change per region. Check: the variant *all land* reproduces
`amoc_effect_<set>.csv` of 6.b.

In [ ]:
pct = lambda a, b: 100 * (a - b) / b if b else np.nan        # as 6.b
PAIRS_ALL = PAIRS + [(run, "ERA5") for run in RUNS]
area_rows, eff_rows, reg_rows, GL = [], [], [], {}
for crop in CROPS:
    f = B6 / f"climate_soil_scenarios_{crop}.nc"
    assert f.exists(), f"missing (run 6.b first): {f}"
    with xr.open_dataset(f) as ds:
        miss = [s for s in SCEN if s not in ds.scenario.values]
        assert not miss, f"{crop}: scenarios {miss} not in 6.b"
        S = {lv: np.nan_to_num(ds[f"suitable_{lv}"].sel(scenario=SCEN).values) == 1 for lv in LEVELS}
        N = ds["novel"].sel(scenario=SCEN).values == 1
    for k, s in enumerate(SCEN):
        for lv in LEVELS:
            for a, w in AV.items():
                area_rows.append({"crop": crop, "scenario": s, "level": lv, "availability": a,
                                  "suitable (Mha)": mha(S[lv][k] * w)})
    for hos, ref in PAIRS_ALL:
        h, o = SCEN.index(hos), SCEN.index(ref)
        both = LAND & ~(N[h] | N[o])
        for lv in LEVELS:
            sh, so = S[lv][h], S[lv][o]
            GL[(crop, f"{hos} - {ref}", lv)] = (sh & ~so & both, so & ~sh & both)
            for a, w in AV.items():
                eff_rows.append({"crop": crop, "pair": f"{hos} - {ref}", "level": lv, "availability": a,
                                 "suitable, reference (Mha)": mha(so * w),
                                 "suitable change, all land (%)": pct(mha(sh * w), mha(so * w)),
                                 "suitable change, assessable in both (%)": pct(mha((sh & both) * w), mha((so & both) * w)),
                                 "gained (Mha)": mha((sh & ~so & both) * w), "lost (Mha)": mha((so & ~sh & both) * w)})
                for reg, rm in REGIONS.items():
                    ao, ah = mha((so & both & rm) * w), mha((sh & both & rm) * w)
                    reg_rows.append({"crop": crop, "pair": f"{hos} - {ref}", "level": lv, "availability": a,
                                     "region": reg, "suitable, reference (Mha)": ao, "suitable change (Mha)": ah - ao,
                                     "suitable change (%)": pct(ah, ao) if ao > 0.5 else np.nan})
    print(f"{crop}: done", flush=True)
AREAS = pd.DataFrame(area_rows).set_index(["crop", "scenario", "level", "availability"])
EFF = pd.DataFrame(eff_rows).set_index(["crop", "pair", "level", "availability"])
REG = pd.DataFrame(reg_rows).set_index(["crop", "pair", "level", "availability", "region"])
AREAS.to_csv(OUT / "areas_availability.csv"); EFF.to_csv(OUT / "amoc_effect_availability.csv")
REG.to_csv(OUT / "amoc_effect_regions_availability.csv")

# check: "all land" = 6.b
f6 = B6 / f"amoc_effect_{SET}.csv"
if f6.exists():
    ref6 = pd.read_csv(f6).set_index(["crop", "pair", "level"])
    mine = EFF.xs("all land", level="availability")
    cols = ["suitable change, all land (%)", "suitable change, assessable in both (%)", "gained (Mha)", "lost (Mha)"]
    common = mine.index.intersection(ref6.index)
    d = (mine.loc[common, cols] - ref6.loc[common, cols]).abs().max()
    print(f"check against 6.b ({len(common)} rows): largest difference per column\n{d.to_string()}")
    assert len(common) and (d.fillna(0) < 1e-6).all(), "the variant 'all land' does not reproduce 6.b"
else:
    print("6.b table not found, check skipped:", f6)

with pd.option_context("display.max_rows", 300, "display.width", 250):
    for lv in LEVELS[::-1]:
        for met in ["suitable change, assessable in both (%)", "gained (Mha)", "lost (Mha)"]:
            t = EFF.xs(lv, level="level")[met].unstack("availability")[list(AV)]
            t = t[t.index.get_level_values("pair").isin([f"{h} - {o}" for h, o in PAIRS])]
            print(f"{met}, {lv} input: hosing - no hosing"); display(t.round(2))
        t = REG.xs((lv, "north of 50 N"), level=("level", "region"))["suitable change (%)"].unstack("availability")[list(AV)]
        print(f"north of 50 N, suitable change (%), {lv} input"); display(t.round(1))

LV = "high" if "high" in LEVELS else LEVELS[0]
h, o = PAIRS[-1]
fig, axes = plt.subplots(1, 2, figsize=(15, 0.6 * len(CROPS) + 2))
colours = ["#4d4d4d", "#1b9e77", "#7570b3", "#d95f02"]
for ax, (src, key, title) in zip(axes, [(EFF, None, "all land (assessable in both)"), (REG, "north of 50 N", "north of 50 N")]):
    for n, a in enumerate(AV):
        vals = []
        for crop in CROPS:
            if key is None:
                vals.append(src.loc[(crop, f"{h} - {o}", LV, a), "suitable change, assessable in both (%)"])
            else:
                vals.append(src.loc[(crop, f"{h} - {o}", LV, a, key), "suitable change (%)"])
        ax.barh(np.arange(len(CROPS)) + (n - 1.5) * 0.2, vals, height=0.19, color=colours[n], label=a)
    ax.axvline(0, color="grey", lw=0.8); ax.set_yticks(range(len(CROPS)))
    ax.set_yticklabels([c.replace("_rf", "") for c in CROPS]); ax.invert_yaxis(); ax.grid(axis="x", alpha=0.3)
    ax.set_title(f"{title}: suitable land change (%)", fontsize=10)
axes[0].legend(fontsize=8)
fig.suptitle(f"Effect of the AMOC weakening (2050 hosing - no hosing, {LV} input) with the availability variants", fontsize=11)
fig.tight_layout(); fig.savefig(OUT / f"amoc_effect_availability_{LV}.png", dpi=100, bbox_inches="tight"); plt.show()

## 4. Where the land gained and lost lies

For each pair, the land gained and lost (cells assessable in both, as 6.b) split by the share of each cell that is
available, protected, built-up (not protected) or tree cover (neither protected nor built-up).

In [ ]:
rows = []
for (crop, pair, lv), (gain, loss) in GL.items():
    for what, m in (("gained", gain), ("lost", loss)):
        tot = mha(m)
        r = {"crop": crop, "pair": pair, "level": lv, "land": what, "total (Mha)": tot}
        for c, w in CATS.items():
            r[f"{c} (Mha)"] = mha(m * w)
            r[f"{c} (%)"] = 100 * mha(m * w) / tot if tot > 0 else np.nan
        rows.append(r)
GLT = pd.DataFrame(rows).set_index(["crop", "pair", "level", "land"])
GLT.to_csv(OUT / "gained_lost_by_land_category.csv")
with pd.option_context("display.max_rows", 300, "display.width", 250):
    for pair in [f"{PAIRS[-1][0]} - {PAIRS[-1][1]}", f"{PAIRS[-1][1]} - ERA5"]:
        print(f"{pair}, {LV} input: share of the land gained and lost in each category (%)")
        display(GLT.xs((pair, LV), level=("pair", "level"))[["total (Mha)", *[f"{c} (%)" for c in CATS]]].round(1))

cc = {"available": "#1a9641", "protected": "#7b3294", "built-up": "#636363", "tree cover (not protected)": "#a6611a"}
pairs_fig = [f"{PAIRS[-1][0]} - {PAIRS[-1][1]}", f"{PAIRS[-1][1]} - ERA5"]
fig, axes = plt.subplots(1, len(pairs_fig), figsize=(8 * len(pairs_fig), 0.75 * len(CROPS) + 2), squeeze=False)
for ax, pair in zip(axes[0], pairs_fig):
    y = 0; labels = []
    for crop in CROPS:
        for what, sign in (("gained", 1), ("lost", -1)):
            left = 0.0
            for c in CATS:
                v = GLT.loc[(crop, pair, LV, what), f"{c} (Mha)"]
                ax.barh(y, sign * v, left=sign * left, color=cc[c], label=c if y == 0 else None, height=0.8)
                left += v
            labels.append(f"{crop.replace('_rf', '')} {what}"); y += 1
    ax.axvline(0, color="k", lw=0.8); ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=8)
    ax.invert_yaxis(); ax.grid(axis="x", alpha=0.3); ax.set_xlabel("Mha (lost to the left)")
    ax.set_title(f"{pair}, {LV} input: land gained and lost by land category", fontsize=10)
axes[0, 0].legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT / f"gained_lost_by_land_category_{LV}.png", dpi=100, bbox_inches="tight"); plt.show()
print("written:", sorted(p.name for p in OUT.iterdir()))

## 5. Reading

To be written after the run on BDAP: (1) the share of protected land in the regions against the Protected Planet
statistics; (2) does the effect of the AMOC weakening (%) keep its sign and size with the availability variants, on all
land and north of 50° N; (3) which share of the land gained and lost is protected or forest, for the AMOC effect and for
the warming; (4) what to report: the potential of 6.b as the main result, the available land next to it.